In [10]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv('tokyo_weather.csv')

df_train = df.iloc[:-30].copy()
df_test = df.iloc[-30:].copy()


df.head(1)

,Date,Average Temperature (°C),Highest Temperature (°C),Highest Temperature (°C) Datetime,Lowest Temperature (°C),Lowest Temperature (°C) Datetime,Total Precipitation (mm),Sunshine Duration (hours),Maximum Snow Depth (cm),Maximum Snow Depth (cm) Datetime,...,Maximum Wind Speed (m/s) Datetime,Maximum Wind Speed (m/s) Direction,Maximum Gust Speed (m/s),Maximum Gust Speed (m/s) Datetime,Maximum Gust Speed (m/s) Direction,Most Frequent Wind Direction (16-point compass),Average Vapor Pressure (hPa),Average Humidity (%),Minimum Relative Humidity (%),Minimum Relative Humidity (%) Datetime
0,2018/6/26,25.7,30.1,2018/6/26 12:38,22.3,2018/6/26 5:37,0.0,9.2,0,NaN,...,2018/6/26 20:04,S,12.4,2018/6/26 21:03,SSW,SE,24.7,75,56,2018/6/26 12:29


In [ ]:
import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

train_data = df_train[[
    "Highest Temperature (°C)",
    "Total Precipitation (mm)"
]].copy()

# Convert to numeric just in case
train_data["Highest Temperature (°C)"] = pd.to_numeric(
    train_data["Highest Temperature (°C)"],
    errors="coerce"
)

train_data["Total Precipitation (mm)"] = pd.to_numeric(
    train_data["Total Precipitation (mm)"],
    errors="coerce"
)

# Remove missing rows
train_data = train_data.dropna().reset_index(drop=True)

# Target variable
y = train_data["Highest Temperature (°C)"]

# ---------------------------------------------------------
# Create exogenous variables
# ---------------------------------------------------------

t = np.arange(len(train_data))

X = pd.DataFrame({
    "Total Precipitation (mm)": train_data["Total Precipitation (mm)"],
    "sin_year": np.sin(2 * np.pi * t / 365),
    "cos_year": np.cos(2 * np.pi * t / 365)
})

# ---------------------------------------------------------
# Time series cross-validation setup
# ---------------------------------------------------------

n_splits = 10
tscv = TimeSeriesSplit(n_splits=n_splits, test_size=1)

predictions = []
actuals = []
fold_numbers = []

# ---------------------------------------------------------
# Cross-validation loop for SARIMAX
# ---------------------------------------------------------

for fold, (train_index, test_index) in enumerate(tscv.split(y), start=1):

    y_train_fold = y.iloc[train_index]
    y_test_fold = y.iloc[test_index]

    X_train_fold = X.iloc[train_index]
    X_test_fold = X.iloc[test_index]

    print(f"\nFold {fold}")
    print("Training rows:", train_index[0], "to", train_index[-1])
    print("Testing row:", test_index[0])

    # -----------------------------------------------------
    # Fit SARIMAX on this training fold
    # -----------------------------------------------------

    sarimax_model = SARIMAX(
        y_train_fold,
        order=(1, 1, 1),
        seasonal_order=(0, 0, 0, 0),
        exog=X_train_fold,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    model_fit = sarimax_model.fit(disp=False)

    # -----------------------------------------------------
    # Forecast the next 1 day
    # -----------------------------------------------------

    forecast_result = model_fit.get_forecast(
        steps=1,
        exog=X_test_fold
    )

    forecast_value = forecast_result.predicted_mean.iloc[0]

    predictions.append(forecast_value)
    actuals.append(y_test_fold.iloc[0])
    fold_numbers.append(fold)

# ---------------------------------------------------------
# Cross-validation results table
# ---------------------------------------------------------

cv_results = pd.DataFrame({
    "Fold": fold_numbers,
    "Actual": actuals,
    "Predicted": predictions
})

cv_results["Error"] = cv_results["Actual"] - cv_results["Predicted"]
cv_results["Absolute Error"] = cv_results["Error"].abs()

display(cv_results)

# ---------------------------------------------------------
# Cross-validation accuracy
# ---------------------------------------------------------

mae_cv = mean_absolute_error(cv_results["Actual"], cv_results["Predicted"])
rmse_cv = np.sqrt(mean_squared_error(cv_results["Actual"], cv_results["Predicted"]))

print("\nCross-validation MAE:", mae_cv)
print("Cross-validation RMSE:", rmse_cv)

# ---------------------------------------------------------
# Fit final SARIMAX model using all training data
# ---------------------------------------------------------

final_model = SARIMAX(
    y,
    order=(1, 1, 1),
    seasonal_order=(0, 0, 0, 0),
    exog=X,
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_fit = final_model.fit(disp=False)

print(final_fit.summary())

# ---------------------------------------------------------
# Get only last 15 training predictions
# ---------------------------------------------------------

n_predictions = 15

prediction_result = final_fit.get_prediction(
    start=len(y) - n_predictions,
    end=len(y) - 1
)

predicted_values = prediction_result.predicted_mean
actual_values = y.iloc[-n_predictions:]

# ---------------------------------------------------------
# Last 15 comparison table
# ---------------------------------------------------------

comparison_table = pd.DataFrame({
    "Actual": actual_values.values,
    "Predicted": predicted_values.values
})

comparison_table["Error"] = comparison_table["Actual"] - comparison_table["Predicted"]
comparison_table["Absolute Error"] = comparison_table["Error"].abs()

display(comparison_table)

# ---------------------------------------------------------
# MAE for last 15 training predictions
# ---------------------------------------------------------

mae_last_15 = mean_absolute_error(
    comparison_table["Actual"],
    comparison_table["Predicted"]
)

print("\nMAE for last 15 training predictions:", mae_last_15)

                                  SARIMAX Results                                   
Dep. Variable:     Highest Temperature (°C)   No. Observations:                 2163
Model:                     SARIMAX(1, 1, 1)   Log Likelihood               -5338.522
Date:                      Wed, 29 Apr 2026   AIC                          10689.045
Time:                              21:10:56   BIC                          10723.112
Sample:                                   0   HQIC                         10701.505
                                     - 2163                                         
Covariance Type:                        opg                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Total Precipitation (mm)    -0.0567      0.002    -25.002      0.000      -0.061      -0.052
sin_year                     5.2379      

,Actual,Predicted,Error,Absolute Error
0,22.0,22.146628,-0.146628,0.146628
1,21.6,25.776003,-4.176003,4.176003
2,25.7,23.894560,1.805440,1.805440
3,25.1,25.473341,-0.373341,0.373341
4,27.2,26.057353,1.142647,1.142647
5,28.8,26.895094,1.904906,1.904906
6,24.1,27.803531,-3.703531,3.703531
7,21.4,23.662687,-2.262687,2.262687
8,28.9,25.207968,3.692032,3.692032
9,25.3,27.987799,-2.687799,2.687799


MAE for last 15 training predictions: 2.0250744159832896
